# FINd — 5-minute integration tutorial

Welcome! This notebook walks through how to use the FINd image hashing library in your own projects. It covers:

1. **Library mode** — import and call from Python
2. **Interpreting hashes** — what a 256-bit hash looks like and how to compare
3. **Near-duplicate detection** — find which images in a folder are the same meme
4. **Threshold selection** — picking the right Hamming distance cutoff for your use case
5. **API mode** — calling the FastAPI service from any HTTP client
6. **Batch hashing for large archives** — processing thousands of images efficiently

If you've already read the `README.md`, this notebook makes everything concrete. If you haven't, this is the friendlier place to start.

## 0. Setup

The cells below assume you've installed the dev requirements (`pip install -r requirements.txt`) and that you have access to the `meme_images/` dataset (see the dataset section in `README.md`).

If you only need the runtime API, install `requirements-api.txt` instead — but this notebook uses `matplotlib` and `pandas` for visualisation, which are dev-only.

In [ ]:
import sys, os
from pathlib import Path

# Make project root importable when running notebook from notebooks/ subdir
_here = Path.cwd()
PROJECT_ROOT = _here.parent if _here.name == 'notebooks' else _here
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

IMG_DIR = PROJECT_ROOT / 'meme_images'
print(f'Project root: {PROJECT_ROOT}')
print(f'Dataset present: {IMG_DIR.exists()} ({len(list(IMG_DIR.glob("*.jpg"))) if IMG_DIR.exists() else 0} images)')

## 1. Library mode — hash a single image

This is the simplest use: import the hasher, call `fromFile`, get back a hash object.

In [ ]:
from FINd_optimized import FINDHasherOptimized

# Create the hasher once at startup — reusable for any number of images.
hasher = FINDHasherOptimized()

# Hash one image
sample_image = next(IMG_DIR.glob('*.jpg'))
print(f'Hashing: {sample_image.name}')

hash_obj = hasher.fromFile(str(sample_image))
print(f'\nHash object: {hash_obj}')
print(f'Type:        {type(hash_obj).__name__}')
print(f'Hex string:  {str(hash_obj)}')
print(f'Length:      {len(str(hash_obj))} characters ({len(str(hash_obj)) * 4} bits)')

**What just happened?** The hasher:

1. Loaded the JPEG and shrank it to ≤512×512 pixels (preserves aspect ratio).
2. Converted RGB → grayscale via the BT.601 luma formula.
3. Smoothed via a box filter (low-pass: removes high-frequency JPEG artifacts before downsampling).
4. Sampled down to 64×64 grid.
5. Took a 16×16 DCT (Discrete Cosine Transform — captures the dominant frequencies).
6. Compared each DCT coefficient to the median; bits = 1 where above, 0 where below.

The result is 256 bits = 64 hex characters. Two visually similar images will have most of these bits matching.

## 2. Compare two images

Hashing one image isn't useful on its own — the value is comparing two. Subtract two hashes to get the **Hamming distance** (number of bit positions that differ, 0–256).

In [ ]:
# Find two images from the same meme cluster (same prefix = same template)
images_in_cluster_0 = sorted(IMG_DIR.glob('0000_*.jpg'))[:2]
img_a, img_b = images_in_cluster_0
print(f'A: {img_a.name}')
print(f'B: {img_b.name}')

hash_a = hasher.fromFile(str(img_a))
hash_b = hasher.fromFile(str(img_b))

print(f'\nA hash: {hash_a}')
print(f'B hash: {hash_b}')

distance = hash_a - hash_b
print(f'\nHamming distance: {distance} bits out of 256')
print(f'Similarity:       {(256 - distance) / 256 * 100:.1f}% bits match')

**Interpretation of the distance**:

| distance range | meaning | use it when |
|---|---|---|
| 0 | bit-exact identical hashes | exact duplicate detection |
| 1–50 | very similar, almost certainly the same content with minor edits | safe to auto-act (block, dedupe) |
| 50–90 | similar — likely related but might be re-cropped, recolored, or substantially edited | flag for human review |
| 90–130 | edge case — could go either way | uncertain, requires context |
| 130+ | unrelated images | filter out, treat as different |

These cutoffs come from ROC analysis on the meme_images dataset (see `accuracy_benchmark.ipynb`); the API surfaces them as `threshold_recommendation` in every response.

## 3. Near-duplicate detection across a folder

Now the realistic use case: you have a folder of images, and you want to find which ones are duplicates / near-duplicates of each other.

We sample 20 images from 4 different meme clusters (so 5 from each), hash all of them, and check pairwise distances.

In [ ]:
import random
random.seed(42)

# Pick 4 random clusters with at least 5 images each
clusters = {}
for f in IMG_DIR.iterdir():
    if f.suffix == '.jpg':
        cluster_id = f.name.split('_')[0]
        clusters.setdefault(cluster_id, []).append(f)

big_clusters = [cid for cid, files in clusters.items() if len(files) >= 5]
chosen = random.sample(big_clusters, 4)
sample_files = []
for cid in chosen:
    sample_files.extend(random.sample(clusters[cid], 5))

print(f'Sampled {len(sample_files)} files from clusters: {chosen}')
print(f'\nFirst few files:')
for f in sample_files[:6]:
    print(f'  {f.name}')

In [ ]:
# Hash all 20 files (~100 ms total at 5 ms each)
import time

t0 = time.perf_counter()
hashes = {f.name: hasher.fromFile(str(f)) for f in sample_files}
elapsed_ms = (time.perf_counter() - t0) * 1000

print(f'Hashed {len(hashes)} images in {elapsed_ms:.0f} ms ({elapsed_ms/len(hashes):.1f} ms/img)')

In [ ]:
# Build pairwise distance matrix
import numpy as np
import pandas as pd

names = list(hashes.keys())
n = len(names)
distances = np.zeros((n, n), dtype=int)
for i in range(n):
    for j in range(n):
        distances[i, j] = hashes[names[i]] - hashes[names[j]]

# Show first 6x6 block of the matrix as a DataFrame for readability
short_names = [n[:14] for n in names[:6]]
df = pd.DataFrame(distances[:6, :6], index=short_names, columns=short_names)
print('Pairwise Hamming distances (first 6 images):')
df

**What you should see**: low distances (0-50) clustered along the diagonal blocks where files share a prefix (`0XXX_`), and high distances (~120) between blocks. That's the signature of "same meme cluster" being captured by the hash.

In [ ]:
# Visualise as a heatmap — same-cluster pairs should form bright blocks on the diagonal
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(distances, cmap='viridis_r', vmin=0, vmax=140)
ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels([n[:8] for n in names], rotation=90, fontsize=8)
ax.set_yticklabels([n[:8] for n in names], fontsize=8)
ax.set_title('Pairwise Hamming distance (lower = more similar)\n5 images each from 4 random meme clusters')
plt.colorbar(im, ax=ax, label='Hamming distance (bits)')
plt.tight_layout()
plt.show()

**Reading the heatmap**: dark squares (low distance) = pairs the algorithm thinks are very similar. The 4 dark 5×5 blocks along the diagonal correspond to the 4 clusters — within-cluster images are correctly identified as similar. The brighter background = pairs from different clusters, correctly identified as different.

## 4. Threshold selection — filter pairs by use case

Once you have a hash database, you usually want to **find duplicates above some similarity threshold**. The right threshold depends on your use case:

In [ ]:
def find_duplicates(hash_dict, threshold):
    """Return all (file_a, file_b, distance) triples where distance <= threshold."""
    pairs = []
    names = list(hash_dict.keys())
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            d = hash_dict[names[i]] - hash_dict[names[j]]
            if d <= threshold:
                pairs.append((names[i], names[j], d))
    return sorted(pairs, key=lambda x: x[2])

# 'high precision' = strict, only flags very-confident duplicates
strict = find_duplicates(hashes, threshold=50)
print(f'High-precision (threshold=50): {len(strict)} pairs flagged')
for a, b, d in strict[:5]:
    print(f'  {d:3d} bits — {a[:14]} ↔ {b[:14]}')

# 'balanced' = moderate, FPR ~1%
balanced = find_duplicates(hashes, threshold=90)
print(f'\nBalanced (threshold=90): {len(balanced)} pairs flagged')

# 'high recall' = lenient, catches more borderline cases
lenient = find_duplicates(hashes, threshold=130)
print(f'High-recall (threshold=130): {len(lenient)} pairs flagged')

**Choosing the threshold**:

- **Content moderation (block uploads)**: use `high_precision` (≤50). False positive = blocking legitimate user content, which is expensive — better to occasionally miss a duplicate than wrongly block.
- **Search & recommendation ("similar images")**: use `balanced` (≤90). Mix of precision and recall fits exploratory search.
- **Deduplication suggestions ("did you mean to upload twice?")**: use `high_recall` (≤130). Showing the user a borderline match and asking is usually fine; missing the match is more annoying.

These three numbers come back in every API response under `threshold_recommendation`.

## 5. API mode — call the running FastAPI service

If your application isn't Python, or if you want to deploy FINd as a microservice, use the HTTP API. Start the server first:

```bash
docker run -d --rm -p 8945:8945 fin/find
# or
python -m api
```

Then call it from any HTTP client. Below we use Python's `httpx` for illustration.

In [ ]:
# This cell ASSUMES the API is running on http://127.0.0.1:8945
# If you haven't started it, this will time out — that's expected.
import httpx

API_URL = 'http://127.0.0.1:8945'

# Probe /health first to confirm the server is up
try:
    r = httpx.get(f'{API_URL}/health', timeout=2.0)
    print(f'Server status: {r.json()}')
    server_up = True
except Exception as e:
    print(f'Server not reachable: {e}')
    print('Start it with: docker run -d --rm -p 8945:8945 fin/find')
    server_up = False

In [ ]:
# If server is up, call /compare
if server_up:
    with open(sample_files[0], 'rb') as f1, open(sample_files[1], 'rb') as f2:
        files = {
            'image1': (sample_files[0].name, f1.read(), 'image/jpeg'),
            'image2': (sample_files[1].name, f2.read(), 'image/jpeg'),
        }
    r = httpx.post(f'{API_URL}/compare', files=files, timeout=10.0)
    response = r.json()
    print(f'HTTP {r.status_code}')
    for key, val in response.items():
        print(f'  {key}: {val}')
else:
    print('(skipped — start the server to run this cell)')

**API response fields**:

- The first three (`image1_hash`, `image2_hash`, `distance`) are the contract per the assignment brief and match the hash you'd get from the Python library byte-for-byte.
- `confidence` is the `bucket_confidence(distance)` label — saves the consumer from learning the cutoffs.
- `image{1,2}_meta` lets downstream pipelines (storage, audit log, recommendation) record metadata without a second HTTP fetch.
- `threshold_recommendation` documents the cutoffs we calibrated from ROC analysis — the consumer can override these for their own use case.

**Interactive testing**: visit `http://127.0.0.1:8945/docs` in a browser. FastAPI auto-generates a Swagger UI from the Pydantic schemas — you can upload images and execute requests right in the browser.

## 6. Batch hashing for large archives

If you have thousands of images to hash (e.g. building an initial database), don't loop over them with the API — just call the library directly. Each hash takes ~5 ms; 10,000 images = ~50 seconds single-threaded.

In [ ]:
# Demo: hash 100 images sequentially, print throughput
batch_files = list(IMG_DIR.glob('*.jpg'))[:100]

t0 = time.perf_counter()
batch_hashes = {f.name: hasher.fromFile(str(f)) for f in batch_files}
elapsed_s = time.perf_counter() - t0

print(f'Hashed {len(batch_files)} images in {elapsed_s:.2f} s')
print(f'Mean per image: {elapsed_s / len(batch_files) * 1000:.1f} ms')
print(f'Throughput:     {len(batch_files) / elapsed_s:.0f} images/sec')
print()
print(f'Projected for full meme_images dataset (55 972 imgs):')
print(f'  Single-threaded: {55972 * elapsed_s / len(batch_files) / 60:.1f} minutes')
print(f'  With 4 workers (multiprocessing.Pool): ~{55972 * elapsed_s / len(batch_files) / 60 / 4:.1f} minutes')

**For larger scale** (millions of images):

- **Multiprocessing**: wrap the hashing call in `concurrent.futures.ProcessPoolExecutor` and process N workers in parallel. The hasher is stateless after construction (`DCT_matrix` is read-only), so each worker can have its own instance.
- **Storage**: hashes are 32 bytes each (256 bits). 1 million images = 32 MB of hash data — fits in memory easily.
- **Index**: for fast nearest-neighbour search at scale, use [Faiss](https://github.com/facebookresearch/faiss) (binary index supports Hamming distance natively) or [hnswlib](https://github.com/nmslib/hnswlib) with custom distance.

These deployment patterns are out of scope for the library itself — but the hash format we produce is compatible with any standard tooling.

## You're done!

You now know how to:

- Hash a single image (`hasher.fromFile(...)`)
- Compare two hashes via Hamming distance (`hash_a - hash_b`)
- Detect near-duplicates across a folder (pairwise distance + threshold)
- Pick the right threshold for your use case (high-precision / balanced / high-recall)
- Call the FastAPI service from any HTTP client
- Process large archives efficiently

For deeper questions:

- **Performance characteristics**: see `notebooks/profile.ipynb` (latency / cProfile / line_profiler / memory / scaling)
- **Comparison vs `imagehash` library**: see `notebooks/accuracy_benchmark.ipynb` (1000 within + 1000 between pairs, FINd vs pHash vs wHash, ROC analysis)
- **Cross-version comparison**: see `notebooks/compare_versions.ipynb`
- **Test suite**: `pytest tests/` (65 tests, ~5 min)

Issues, feedback, or production-deployment questions — see the **Get in touch with FIN** section in `README.md`.